In [2]:
!pip install -q "datasets<3.0" transformers peft bitsandbytes accelerate torch

In [3]:
from huggingface_hub import login
login()  # will prompt for your token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

model_id   = "meta-llama/Meta-Llama-3-8B-Instruct"
adapter_id = "oLittle-five/llama3-8b-wikisql-qlora"   # v1 adapter

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)

model = PeftModel.from_pretrained(base_model, adapter_id)
model.eval()
print(f"Done! Memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

Done! Memory used: 6.2 GB


In [5]:
import json
from datasets import load_dataset
from google.colab import files

ds = load_dataset("Salesforce/wikisql", trust_remote_code=True)
examples = list(ds['test'].select(range(500)))

uploaded = files.upload()  # upload base_model_results.json

with open("base_model_results.json") as f:
    base_results = json.load(f)

base_predictions = base_results["predictions"]
print(f"Loaded {len(base_predictions)} base model predictions")
print(f"Base model execution accuracy: {base_results['execution_accuracy']:.1%}")

Generating test split:   0%|          | 0/15878 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/8421 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/56355 [00:00<?, ? examples/s]

Saving base_model_results.json to base_model_results.json
Loaded 500 base model predictions
Base model execution accuracy: 37.2%


In [6]:
import re

# ── Stop tokens ────────────────────────────────────────────────────────────────
eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")
stop_token_ids = [tokenizer.eos_token_id, eot_id]

# ── Chat template prefix — matches exactly what SFTTrainer added during v1 training
CHAT_PREFIX = "<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n"


def fix_column_names(sql: str, columns: list) -> str:
    """Wrap column name references in backticks, handling underscore variants."""
    for col in sorted(columns, key=len, reverse=True):
        if not col:
            continue
        quoted = f'`{col}`'
        underscore_variant = re.sub(r'[\s\-–]+', '_', col)
        variants = [col]
        if underscore_variant.lower() != col.lower():
            variants.append(underscore_variant)
        for variant in variants:
            if re.search(re.escape(quoted), sql, re.IGNORECASE):
                break
            if re.search(re.escape(variant), sql, re.IGNORECASE):
                sql = re.sub(re.escape(variant), quoted, sql, flags=re.IGNORECASE)
                break
    return sql


def clean_wikisql(sql: str) -> str:
    """Enforce WikiSQL grammar: no OR, ORDER BY, LIMIT, IS NOT NULL, !="""
    sql = re.sub(r'\s+OR\s+.+$',             '', sql, flags=re.IGNORECASE).strip()
    sql = re.sub(r'\s+ORDER\s+BY\s+.+$',     '', sql, flags=re.IGNORECASE).strip()
    sql = re.sub(r'\s+LIMIT\s+.+$',          '', sql, flags=re.IGNORECASE).strip()
    sql = re.sub(r"\s+AND\s+`[^`]+`\s+IS\s+NOT\s+NULL", '', sql, flags=re.IGNORECASE).strip()
    sql = re.sub(r"""\s+AND\s+`[^`]+`\s*!=\s*['"]?['"]?""", '', sql, flags=re.IGNORECASE).strip()
    return sql


def generate_sql(question: str, columns: list, types: list) -> str:
    """Generate SQL using v1 adapter with chat template prefix at inference."""
    col_defs = ", ".join(f"{name} ({dtype})" for name, dtype in zip(columns, types))
    raw_prompt = (
        f"### Input:\n"
        f"Columns: {col_defs}\n\n"
        f"Question: {question}\n\n"
        f"### SQL:\n"
    )
    # Prepend chat prefix to match v1 training format
    full_prompt = CHAT_PREFIX + raw_prompt

    inputs = tokenizer(full_prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            repetition_penalty=1.3,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=stop_token_ids,
        )
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    if "```" in response:
        lines = response.split('\n')
        response = ' '.join(l for l in lines if not l.strip().startswith('```')).strip()

    for line in response.split('\n'):
        line = line.strip()
        line = re.sub(r'\bassistant\b', '', line, flags=re.IGNORECASE).strip()
        if line.upper().startswith('SELECT'):
            line = line.split(';')[0].strip()
            if '--' in line: line = line[:line.index('--')].strip()
            if '###' in line: line = line[:line.index('###')].strip()
            line = re.sub(r'\s+LIMIT\s+.*$', '', line, flags=re.IGNORECASE).strip()
            line = re.sub(r'\bFROM\s+\w+\b', 'FROM table', line, flags=re.IGNORECASE)
            line = fix_column_names(line, columns)
            line = re.sub(r"\s+AND\s+`[^`]+`\s*=\s*''", '', line, flags=re.IGNORECASE)
            return line

    return response.split('\n')[0].split(';')[0].strip()


# Sanity check on 3 examples
print("=== Sanity check ===\n")
for i in range(3):
    ex = ds['test'][i]
    pred = generate_sql(ex['question'], ex['table']['header'], ex['table']['types'])
    pred = clean_wikisql(pred)
    print(f"Q:    {ex['question']}")
    print(f"Pred: {pred}")
    print(f"Gold: {ex['sql']['human_readable']}")
    print()

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


=== Sanity check ===

Q:    What is terrence ross' nationality
Pred: SELECT `Nationality` FROM table WHERE `Player` = 'Terrence Ross'
Gold: SELECT Nationality FROM table WHERE Player = Terrence Ross

Q:    What clu was in toronto 1995-96
Pred: SELECT `School/Club Team` FROM table WHERE `Years in Toronto` = '1995-96'
Gold: SELECT School/Club Team FROM table WHERE Years in Toronto = 1995-96

Q:    which club was in toronto 2003-06
Pred: SELECT `School/Club Team` FROM table WHERE `Years in Toronto` = '2003-06'
Gold: SELECT School/Club Team FROM table WHERE Years in Toronto = 2003-06



In [7]:
from tqdm.notebook import tqdm

ft_predictions = []
for example in tqdm(examples):
    pred = generate_sql(
        example['question'],
        example['table']['header'],
        example['table']['types']
    )
    pred = clean_wikisql(pred)
    ft_predictions.append(pred)

print(f"Generated {len(ft_predictions)} predictions")
print("\nSample predictions:")
for i in range(3):
    print(f"  {i+1}. {ft_predictions[i]}")

  0%|          | 0/500 [00:00<?, ?it/s]

Generated 500 predictions

Sample predictions:
  1. SELECT `Nationality` FROM table WHERE `Player` = 'Terrence Ross'
  2. SELECT `School/Club Team` FROM table WHERE `Years in Toronto` = '1995-96'
  3. SELECT `School/Club Team` FROM table WHERE `Years in Toronto` = '2003-06'


In [8]:
import sqlite3

def build_db(table):
    conn = sqlite3.connect(":memory:")
    cursor = conn.cursor()
    headers = table["header"]
    types = table["types"]
    type_map = {"text": "TEXT", "number": "REAL", "real": "REAL"}
    col_defs = ", ".join(f'`{col}` {type_map.get(t, "TEXT")}' for col, t in zip(headers, types))
    cursor.execute(f"CREATE TABLE data ({col_defs})")
    for row in table["rows"]:
        converted = []
        for val, col_type in zip(row, types):
            if col_type in ("real", "number"):
                try:
                    converted.append(float(str(val).replace(",", "")))
                except:
                    converted.append(val)
            else:
                converted.append(val)
        placeholders = ", ".join(["?"] * len(converted))
        cursor.execute(f"INSERT INTO data VALUES ({placeholders})", converted)
    conn.commit()
    return conn

def execute_sql(table, sql):
    sql = sql.replace("FROM table", "FROM data")
    try:
        conn = build_db(table)
        cursor = conn.cursor()
        cursor.execute(sql)
        results = cursor.fetchall()
        conn.close()
        return results, None
    except Exception as e:
        return [], str(e)

def normalize_result(result):
    return sorted([str(row) for row in result])

def build_sql_string(sql, columns, types):
    AGG_OPS = ["", "MAX", "MIN", "COUNT", "SUM", "AVG"]
    COND_OPS = ["=", ">", "<"]
    agg = AGG_OPS[sql["agg"]]
    sel_col = columns[sql["sel"]]
    select_clause = f"SELECT {agg}(`{sel_col}`)" if agg else f"SELECT `{sel_col}`"
    where_clauses = []
    for col_idx, op_idx, cond in zip(
        sql["conds"]["column_index"],
        sql["conds"]["operator_index"],
        sql["conds"]["condition"],
    ):
        col = columns[col_idx]
        op = COND_OPS[op_idx]
        col_type = types[col_idx]
        if col_type in ("real", "number"):
            cleaned = str(cond).replace(",", "")
            try:
                float(cleaned)
                where_clauses.append(f"`{col}` {op} {cleaned}")
            except:
                where_clauses.append(f"`{col}` {op} '{str(cond).replace(chr(39), chr(39)*2)}'")
        else:
            escaped = str(cond).replace("'", "''")
            where_clauses.append(f"`{col}` {op} '{escaped}'")
    where_clause = " WHERE " + " AND ".join(where_clauses) if where_clauses else ""
    return select_clause + " FROM table" + where_clause

ft_total         = len(ft_predictions)
ft_correct       = 0
ft_syntax_errors = 0

for pred, example in zip(ft_predictions, examples):
    table    = example["table"]
    gold_sql = build_sql_string(example["sql"], table["header"], table["types"])
    pred_results, pred_error = execute_sql(table, pred)
    gold_results, _          = execute_sql(table, gold_sql)
    if pred_error:
        ft_syntax_errors += 1
    elif normalize_result(pred_results) == normalize_result(gold_results):
        ft_correct += 1

print("=" * 62)
print(f"{'Method':<38} {'Exec Acc':>10} {'Syntax Err':>12}")
print("=" * 62)
print(f"{'Base Llama-3-8B':<38} {'36.8%':>10} {'22.6%':>12}")
print(f"{'v1 original (wrong eval format)':<38} {'25.4%':>10} {'27.4%':>12}")
print(f"{'v2 (raw format, corrected eval)':<38} {'29.0%':>10} {'30.2%':>12}")
print(f"{'v1 fixed (chat template at inference)':<38} {ft_correct/ft_total:>10.1%} {ft_syntax_errors/ft_total:>12.1%}")
print("=" * 62)
improvement = ft_correct / ft_total - 0.368
print(f"\nv1-fixed vs Baseline: {improvement:+.1%}")

Method                                   Exec Acc   Syntax Err
Base Llama-3-8B                             36.8%        22.6%
v1 original (wrong eval format)             25.4%        27.4%
v2 (raw format, corrected eval)             29.0%        30.2%
v1 fixed (chat template at inference)       51.4%         6.0%

v1-fixed vs Baseline: +14.6%


In [9]:
from google.colab import files
import json

v1_fixed_results = {
    "model":               "meta-llama/Meta-Llama-3-8B-Instruct",
    "adapter":             "oLittle-five/llama3-8b-wikisql-qlora",
    "version":             "v1_fixed",
    "eval_format":         "chat_template_at_inference",
    "dataset":             "wikisql",
    "split":               "test",
    "n_examples":          ft_total,
    "execution_accuracy":  ft_correct / ft_total,
    "syntax_error_rate":   ft_syntax_errors / ft_total,
    "correct":             ft_correct,
    "syntax_errors":       ft_syntax_errors,
    "predictions":         ft_predictions,
}

with open("v1_fixed_results.json", "w") as f:
    json.dump(v1_fixed_results, f, indent=2)

files.download("v1_fixed_results.json")
print("Downloaded!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded!
